# Results Visualization
Visualizes all training results across:
- **ADHD-200** (replication of Zou et al. 2017)
- **ABIDE** (generalization to autism classification)

Reads incrementally-saved `results.txt` files so it works even while training is still running.

In [1]:
import ast
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Paths to results directories.
ADHD_DIR  = Path('../outputs_oscar')   # ADHD-200 results from Oscar
ABIDE_DIR = Path('../outputs_abide')   # ABIDE results from Oscar

# Paper targets from Zou et al. 2017 Table III.
PAPER_TARGETS = {
    'falff': 62.06,
    'reho':  60.27,
    'gm':    65.43,
    'falff_gm_multi': 69.15,
}

COLORS = {
    'falff': 'C0',
    'reho':  'C3',
    'gm':    'C1',
    'falff_gm_multi': 'C2',
}

print('Setup done.')

Setup done.


In [ ]:
def load_results(base_dir, experiments):
    """
    Load results.txt for each experiment label.
    Returns a dict: {label: {mean_val_acc, variance, best_run, repeats_done, n_repeats, all_repeat_means}}
    """
    results = {}
    for label in experiments:
        path = base_dir / label / 'results.txt'
        if not path.exists():
            print(f'  {label}: not found yet')
            continue
        d = {}
        with open(path) as f:
            for line in f:
                if ':' not in line:
                    continue
                key, val = line.split(':', 1)
                key, val = key.strip(), val.strip()
                if key == 'mean_val_acc':
                    d['mean_val_acc'] = float(val.replace('%', ''))
                elif key == 'variance':
                    d['variance'] = float(val)
                elif key == 'best_run':
                    d['best_run'] = float(val.replace('%', ''))
                elif key == 'repeats_done':
                    done, total = val.split('/')
                    d['repeats_done'] = int(done)
                    d['n_repeats'] = int(total)
                elif key == 'all_repeat_means':
                    # FIX: Scrub the numpy wrapper text out before evaluating
                    clean_val = val.replace('np.float64(', '').replace(')', '')
                    d['all_repeat_means'] = ast.literal_eval(clean_val)
        d['label'] = label
        results[label] = d
        status = f"{d.get('repeats_done','?')}/{d.get('n_repeats','?')} repeats"
        print(f'  {label:20s}: {d.get("mean_val_acc","?")}%  ({status})')
    return results

print('ADHD-200 results:')
adhd = load_results(ADHD_DIR, ['falff', 'reho', 'gm', 'falff_gm_multi'])

print('\nABIDE results:')
abide = load_results(ABIDE_DIR, ['falff', 'reho', 'gm', 'falff_gm_multi'])

ADHD-200 results:
  falff: not found yet
  reho: not found yet
  gm: not found yet
  falff_gm_multi: not found yet

ABIDE results:


ValueError: malformed node or string on line 1: <ast.Call object at 0x113aec460>

## 1. Summary Table — ADHD-200 vs Paper

In [ ]:
# Compare our ADHD-200 results to Zou et al. 2017 Table III.
# Δ > 0 means we beat the paper.
rows = []
for label, d in adhd.items():
    ours   = d.get('mean_val_acc', float('nan'))
    paper  = PAPER_TARGETS.get(label, float('nan'))
    status = f"{d.get('repeats_done','?')}/{d.get('n_repeats','?')}"
    rows.append({
        'Experiment':   label,
        'Progress':     status,
        'Our acc (%)':  f'{ours:.2f}',
        'Paper (%)':    f'{paper:.2f}',
        'Δ':            f'{ours - paper:+.2f}',
        'Best run (%)': f"{d.get('best_run', float('nan')):.2f}",
        'Variance':     f"{d.get('variance', float('nan')):.2f}",
    })

df = pd.DataFrame(rows)
print('ADHD-200 vs Zou et al. 2017:')
print(df.to_string(index=False))

## 2. Summary Table — ABIDE (Autism)

In [ ]:
# ABIDE has no paper target for this exact model, so we compare against chance (50%).
rows = []
for label, d in abide.items():
    ours   = d.get('mean_val_acc', float('nan'))
    status = f"{d.get('repeats_done','?')}/{d.get('n_repeats','?')}"
    rows.append({
        'Experiment':   label,
        'Progress':     status,
        'Our acc (%)':  f'{ours:.2f}',
        'vs Chance':    f'{ours - 50:+.2f}',
        'Best run (%)': f"{d.get('best_run', float('nan')):.2f}",
        'Variance':     f"{d.get('variance', float('nan')):.2f}",
    })

df2 = pd.DataFrame(rows)
print('ABIDE autism classification:')
print(df2.to_string(index=False))

## 3. Bar Chart — Our Accuracy vs Paper Targets (ADHD-200)

In [ ]:
# Side-by-side bars: our result (solid) vs paper target (hatched).
# Only includes experiments that have completed at least 1 repeat.
labels  = [l for l in ['falff', 'reho', 'gm', 'falff_gm_multi'] if l in adhd]
ours    = [adhd[l]['mean_val_acc'] for l in labels]
paper   = [PAPER_TARGETS.get(l, 0) for l in labels]
x       = np.arange(len(labels))
width   = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, ours,  width, label='Ours',  color=[COLORS[l] for l in labels])
bars2 = ax.bar(x + width/2, paper, width, label='Paper', color=[COLORS[l] for l in labels],
               alpha=0.4, hatch='//')

# Annotate bar heights.
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, color='gray')

ax.axhline(50, color='red', linestyle='--', linewidth=1, label='Chance (50%)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean val accuracy (%)')
ax.set_title('ADHD-200: Our Results vs Zou et al. 2017')
ax.set_ylim(40, 100)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs_oscar/bar_adhd_vs_paper.png', dpi=150)
plt.show()

## 4. Per-repeat Accuracy — ADHD-200

In [ ]:
# Each point = mean val acc across 4 folds for that repeat.
# Dashed lines = paper targets. Tight clustering = stable model.
fig, ax = plt.subplots(figsize=(12, 5))

for label, d in adhd.items():
    vals = d.get('all_repeat_means', [])
    if not vals:
        continue
    c = COLORS[label]
    ax.plot(range(1, len(vals)+1), vals, marker='.', color=c, linewidth=1, label=label)
    if label in PAPER_TARGETS:
        ax.axhline(PAPER_TARGETS[label], linestyle='--', color=c, alpha=0.4)

ax.axhline(50, color='red', linestyle=':', linewidth=1, label='Chance')
ax.set_xlabel('Repeat')
ax.set_ylabel('Mean val accuracy (%)')
ax.set_title('ADHD-200 — accuracy per repeat (dashed = paper target)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs_oscar/per_repeat_adhd.png', dpi=150)
plt.show()

## 5. Per-repeat Accuracy — ABIDE

In [ ]:
# Same plot for ABIDE. No paper targets — compare against 50% chance.
fig, ax = plt.subplots(figsize=(12, 5))

for label, d in abide.items():
    vals = d.get('all_repeat_means', [])
    if not vals:
        continue
    c = COLORS[label]
    ax.plot(range(1, len(vals)+1), vals, marker='.', color=c, linewidth=1, label=label)

ax.axhline(50, color='red', linestyle='--', linewidth=1.5, label='Chance (50%)')
ax.set_xlabel('Repeat')
ax.set_ylabel('Mean val accuracy (%)')
ax.set_title('ABIDE — accuracy per repeat (autism classification)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs_abide/per_repeat_abide.png', dpi=150)
plt.show()

## 6. ADHD-200 vs ABIDE Side-by-Side

In [ ]:
# Compare the same model's performance on two different disorders.
# If ABIDE acc > 50%, brain patterns learned transfer across disorders.
shared = [l for l in ['falff', 'reho', 'gm', 'falff_gm_multi']
          if l in adhd and l in abide]

adhd_accs  = [adhd[l]['mean_val_acc']  for l in shared]
abide_accs = [abide[l]['mean_val_acc'] for l in shared]
x = np.arange(len(shared))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, adhd_accs,  width, label='ADHD-200', color='steelblue')
ax.bar(x + width/2, abide_accs, width, label='ABIDE',    color='salmon')

for i, (a, b) in enumerate(zip(adhd_accs, abide_accs)):
    ax.text(i - width/2, a + 0.5, f'{a:.1f}%', ha='center', fontsize=8)
    ax.text(i + width/2, b + 0.5, f'{b:.1f}%', ha='center', fontsize=8)

ax.axhline(50, color='red', linestyle='--', linewidth=1, label='Chance (50%)')
ax.set_xticks(x)
ax.set_xticklabels(shared)
ax.set_ylabel('Mean val accuracy (%)')
ax.set_title('Same model: ADHD-200 (ADHD) vs ABIDE (Autism)')
ax.set_ylim(40, 100)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs_oscar/adhd_vs_abide.png', dpi=150)
plt.show()

## 7. Accuracy Distribution (Box Plot)

In [ ]:
# Box plots show spread across 50 repeats — how consistent is the model?
# Narrow box = stable; wide box = high variance across random splits.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (dataset_name, dataset) in zip(axes, [('ADHD-200', adhd), ('ABIDE', abide)]):
    data   = [dataset[l]['all_repeat_means'] for l in dataset if dataset[l].get('all_repeat_means')]
    labels = [l for l in dataset if dataset[l].get('all_repeat_means')]
    if not data:
        ax.set_title(f'{dataset_name} — no data yet')
        continue
    bp = ax.boxplot(data, patch_artist=True, notch=False)
    for patch, label in zip(bp['boxes'], labels):
        patch.set_facecolor(COLORS[label])
        patch.set_alpha(0.6)
    ax.axhline(50, color='red', linestyle='--', linewidth=1, label='Chance')
    ax.set_xticklabels(labels, rotation=15)
    ax.set_ylabel('Mean val accuracy (%)')
    ax.set_title(f'{dataset_name} — accuracy distribution across repeats')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs_oscar/boxplot_all.png', dpi=150)
plt.show()